<a href="https://colab.research.google.com/github/sarangis/python_learning/blob/main/M4/Additional_NB_01_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification Programme in Agentic and Generative AI
## A Programme by IISc and TalentSprint
### Additional Notebook: **Prompt Engineering**

(Ungraded)

## Objective

Experience how different prompt engineering techniques can shape the quality, accuracy, and usefulness of AI outputs. The goal is to highlight how thoughtful prompt design enhances reasoning, creativity, and control compared to generic prompting.

## Instructions

- First, get familiar with different Prompting Techniques by following the steps given in the notebook.

- Then, complete the ungraded practice task given at the end of the notebook.


# I. Getting Started with different Prompting Techniques

## Information

**Prompt** is the input you give to an AI model—a piece of text, instruction, or query that tells the model what you want it to do.

---

**Prompt Engineering** is the practice of carefully designing, crafting, and refining the ***input*** ("prompts") given to generative AI models to achieve desired ***outputs***. It involves using clear language, context, and instructions to guide the AI toward producing relevant, accurate, and specific responses, much like providing a roadmap or detailed instructions to someone.

---

**Elements of a Prompt**

A prompt contains any of the following elements:

- **Instruction** - a specific task or instruction you want the model to perform

- **Context** - external information or additional context that can steer the model to better responses

- **Input Data** - the input or question that we are interested to find a response for

- **Output Indicator** - the type or format of the output.

---

## Groq API

**What is Groq?**

**Groq** is a technology company that builds **specialized AI hardware and software** designed to run large language models (LLMs) and other AI models **extremely fast and efficiently**, especially for **real-time inference**.

This notebook demonstrates how to interact with the **OpenAI and Llama models** using Groq API.

It covers the necessary steps from installing the required packages and setting up authentication to making a chat completion request and processing the response. The example prompt is designed to generate a detailed essay on a specific topic.



**Read the Groq API Key**

Using the Groq API Key, you will have access to the OpenAI and Llama model, free of cost, under the [Free-tier](https://console.groq.com/docs/rate-limits#rate-limits).

This cell retrieves the Groq API key securely stored in Google Colab's Secrets (see 'key' icon on the leftmost panel of this notebook) and sets it as an environment variable `GROQ_API_KEY`. This is a recommended practice to avoid exposing your API key directly in the code.


* Go to https://console.groq.com/, and setup a Free account.

* Create a new API by visiting: https://console.groq.com/keys

* Save the key in Google Colab's Secrets
    ```
    Secret Name: GROQ_API_KEY
    Secret Value: Paste your Groq api key
    ```

* Read the key and save as an environment variable `GROQ_API_KEY`

In [ ]:
# Save the key in Colab's Secrets then load from there

import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

**Create Groq Client**

This cell initializes the Groq client using the API key stored in the environment variable. This client object will be used to make API calls to OpenAI model.


In [ ]:
# Install Groq library
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00


In [ ]:
from groq import Groq

# Initialize Groq client
groq_client = Groq()

## **1) Zero-Shot Prompting**

Large language models (LLMs) today, such as GPT-3.5 Turbo, GPT-4, and Claude 3, are tuned to follow instructions and are trained on large amounts of data. Large-scale training makes these models capable of performing some tasks in a "zero-shot" manner. Zero-shot prompting means that the prompt used to interact with the model won't contain examples or demonstrations. **The zero-shot prompt directly instructs the model to perform a task without any additional examples to steer it.**

Here is one of example for text classification:

Prompt:

```
Classify the text into neutral, negative or positive.
Text: I think the vacation is okay.
Sentiment:
```

---
---

In [ ]:
prompt = """Classify the text into neutral, negative or positive.
Text: I think the vacation is okay.
Sentiment:
"""

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response)
print(response.choices[0].message.content)

ChatCompletion(id='chatcmpl-8d84f454-603f-46c5-a149-023ed2688c98', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Sentiment: neutral', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to classify sentiment: neutral, negative, positive. Text: "I think the vacation is okay." That is neutral sentiment. So output "Sentiment: neutral". The format: They wrote "Sentiment:" then likely a word. We\'ll output "Sentiment: neutral".', tool_calls=None))], created=1774786596, model='openai/gpt-oss-20b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_228717f27c', usage=CompletionUsage(completion_tokens=71, prompt_tokens=94, total_tokens=165, completion_time=0.100859415, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=58), prompt_time=0.007102466, prompt_tokens_details=None, queue_time=0.018424984, total_time=0.107961881), usage_breakd

When zero-shot doesn't work, it's recommended to provide demonstrations or examples in the prompt which leads to few-shot prompting.

## **2) Few-Shot Prompting**

While large-language models demonstrate remarkable zero-shot capabilities, they still fall short on more complex tasks when using the zero-shot setting. Few-shot prompting can be used as a technique to enable in-context learning where we provide demonstrations in the prompt to steer the model to better performance. The demonstrations serve as conditioning for subsequent examples where we would like the model to generate a response.

Here is one of example for text classification:

Prompt:

```
This is awesome! // Positive
This is bad! // Negative
Wow that movie was rad! // Positive
What a horrible show! //
```


In [ ]:
prompt = """This is awesome! // Positive
This is bad! // Negative
Wow that movie was rad! // Positive
What a horrible show! //
"""

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Positive
Negative
Positive
Negative


Standard few-shot prompting works well for many tasks but is still not a perfect technique, especially when dealing with more complex reasoning tasks.

When zero-shot prompting and few-shot prompting are not sufficient, it might mean that whatever was learned by the model isn't enough to do well at the task. From here it is recommended to start thinking about fine-tuning your models or experimenting with more advanced prompting techniques.

## **3) Chain-of-Thought (CoT) Prompting**

<img src='https://www.promptingguide.ai/_next/image?url=%2F_next%2Fstatic%2Fmedia%2Fcot.1933d9fe.png&w=1080&q=75' width=800px>

[Image Source](https://arxiv.org/abs/2201.11903)


Introduced in [Wei et al. (2022)](https://arxiv.org/abs/2201.11903), chain-of-thought (CoT) prompting enables complex reasoning capabilities through intermediate reasoning steps. You can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

For example:

Prompt:

```
I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman.
I then went and bought 5 more apples and ate 1. How many apples did I remain with?
Let's think step by step.
```


In [ ]:
prompt = """I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman.
I then went and bought 5 more apples and ate 1. How many apples did I remain with?
Let's think step by step.
"""

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Let’s track the apples one step at a time.

| Step | Action | Apples before | Apples after |
|------|--------|---------------|--------------|
| 1 | Start | 0 | 0 |
| 2 | Bought 10 apples | 0 | **10** |
| 3 | Gave 2 to neighbor | 10 | **8** |
| 4 | Gave 2 to repairman | 8 | **6** |
| 5 | Bought 5 more apples | 6 | **11** |
| 6 | Ate 1 apple | 11 | **10** |

So after all the buying, giving, and eating, you’re left with **10 apples**.


## **4) System Prompts** $\quad$ (Assigning a Role)

A system prompt is a hidden or background instruction given to an AI model that sets the rules, behavior, or persona of the AI before it responds to user queries.

- It’s not usually visible to the end-user.
- It defines how the AI should think, respond, and behave.
- It acts like the “ground rules” or operating instructions for the AI.

**Difference from User Prompt**

- **System Prompt:** Defines the AI’s role, style, and boundaries. (invisible, set by developers or platform)
- **User Prompt:** The actual question or instruction given by the user.

**Why It Matters**

- Ensures consistent AI behavior across users.
- Prevents harmful, biased, or off-topic outputs.
- Useful for persona-based AI (e.g., tutor, lawyer, doctor assistant).


For Example:

System Prompt:
```
You are a very smart physics professor.
You are great at answering questions about physics in a concise and easy to understand manner.
When you don't know the answer to a question you admit that you don't know.
Start your answer with `Subject:Physics :`
```

Prompt:
```
What is black body radiation?
```

In [ ]:
system_prompt = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.
Start your answer with `Subject:Physics :`
"""

prompt = "What is black body radiation?"

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

**Subject:Physics :**  
A **black body** is an idealized object that *absorbs* all electromagnetic radiation that strikes it, regardless of wavelength or angle. Because it absorbs everything, it also emits the maximum possible amount of radiation for its temperature.  

- **Emission spectrum**: The energy radiated per unit area, per unit time, per unit wavelength (or frequency) depends only on the temperature \(T\).  
- **Planck’s law** gives the spectral distribution:
  \[
  B(\lambda,T)=\frac{2hc^{2}}{\lambda^{5}}\frac{1}{e^{hc/(\lambda k_{B}T)}-1}
  \]
  (or the equivalent in frequency form).  
- **Total power**: Integrating over all wavelengths gives the Stefan–Boltzmann law:
  \[
  P = \sigma T^{4},\qquad \sigma=\frac{2\pi^{5}k_{B}^{4}}{15h^{3}c^{2}}\approx5.67\times10^{-8}\ \text{W\,m}^{-2}\text{K}^{-4}.
  \]
- **Real objects** approximate a black body when they are *black* and have a surface rough enough that incident photons are trapped and re‑emitted many times before escaping

In [ ]:
prompt2 = "What is Vibe Coding?"

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt2}
    ]
)

print(response.choices[0].message.content)

Subject:Physics :  
Vibe Coding refers to using **Vibe**, an open‑source, full‑stack framework written in TypeScript that lets developers build modern web applications with a clean, type‑safe architecture.  It combines several best‑in‑class libraries under a single cohesive stack:

| Layer | Core Components | What It Does |
|-------|-----------------|--------------|
| **Server** | Fastify, Prisma, WebSocket | Handles HTTP requests, database access, real‑time messaging |
| **Client** | React (or any framework), Vite | Renders the UI, supports hot‑module replacement |
| **Shared** | TypeScript, Zod | Ensures type safety across the whole project, validates data on both sides |
| **Build** | Vite + esbuild | Fast bundling and code splitting for both server and client |

### Key Features
- **Type safety everywhere** – shared schema definitions mean the compiler catches mismatches before runtime.
- **Zero‑config or minimal config** – out‑of‑the‑box support for routing, authentication, and da

## **5) Prompt Chaining**

To improve the reliability and performance of LLMs, one of the important prompt engineering techniques is to ***break tasks into its subtasks***. Once those subtasks have been identified, the LLM is prompted with a subtask and then its ***response is used as input to another prompt***. This is what's referred to as prompt chaining, where a task is split into subtasks with the idea to create a chain of prompt operations.

Prompt chaining is useful to accomplish complex tasks which an LLM might struggle to address if prompted with a very detailed prompt. In prompt chaining, chain prompts perform transformations or additional processes on the generated responses before reaching a final desired state.

Besides achieving better performance, prompt chaining helps to boost the transparency of your LLM application, increases controllability, and reliability. This means that you can debug problems with model responses much more easily and analyze and improve performance in the different stages that need improvement.

Prompt chaining is particularly useful when building LLM-powered conversational assistants and improving the personalization and user experience of your applications.


**For Example:** Let us consider an example to see how a large task—writing a blog post on remote work—can be broken into smaller steps using Prompt Chaining.

- First, the AI generates a blog outline,
- then expands one section into detailed text,
- summarizes that section into concise bullet points, and
- finally creates a persuasive call-to-action.

This step-by-step approach ensures clarity, structure, and refined outputs instead of trying to get everything in a single prompt.


In [ ]:
# Step 1 – Generate an Outline

outline_prompt = "Create a 5-point outline for a blog post on the benefits of remote work."

outline = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": outline_prompt}
    ]
).choices[0].message.content

print(outline)

**5‑Point Outline for a Blog Post: “The Top Benefits of Remote Work”**

1. **Hook & Context**  
   * Start with a striking statistic or anecdote (e.g., “85% of employees say they’d work remotely if offered”).  
   * Briefly explain why remote work has become mainstream (pandemic, tech, changing employee expectations).  
   * State the post’s purpose: to highlight the key advantages that make remote work attractive to both employees and employers.

2. **Increased Productivity & Flexibility**  
   * *Eliminate commutes:* save 30‑60 minutes daily → more focus time.  
   * *Self‑paced work schedules:* align work hours with personal peak‑performance periods.  
   * *Fewer office distractions:* data showing fewer interruptions and higher deep‑work output.

3. **Expanded Talent Pool & Cost Savings**  
   * *Access global talent:* hire best fit regardless of geographic constraints.  
   * *Reduced overhead:* lower office rent, utilities, and furniture costs.  
   * *Lower turnover:* happier wo

In [ ]:
# Step 2 – Expand a Section

expand_prompt = "Expand point #2 into a 120-word explanation with real-life examples."

point2_explaination = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": outline_prompt},
        {"role": "assistant", "content": outline},
        {"role": "user", "content": expand_prompt},
    ]
).choices[0].message.content

print(point2_explaination)

**Increased Productivity & Flexibility**  
Remote work eliminates daily commutes, freeing 30–60 minutes that employees can use for focused tasks. In a 2023 Gartner survey, 77 % of remote workers reported a productivity boost, citing fewer office interruptions. Companies like Basecamp have adopted “no‑meeting” days, letting teams dive into deep work and report a 12 % rise in project delivery speed. Flexible schedules also empower employees to work during their peak concentration hours. For example, a data‑analytics team in Berlin shifted to a compressed 4‑day week, and each analyst completed 18 % more code reviews while maintaining work‑life balance. These real‑life outcomes show that flexibility directly translates into higher output and job satisfaction. Even large enterprises like Microsoft and Salesforce report similar gains, citing improved employee morale today.


In [ ]:
# Step 3 – Create a Summary

summary_prompt = "Summarize this section in 2 bullet points for a presentation slide."

point2_summary = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": outline_prompt},
        {"role": "assistant", "content": outline},
        {"role": "user", "content": expand_prompt},
        {"role": "assistant", "content": point2_explaination},
        {"role": "user", "content": summary_prompt},
    ]
).choices[0].message.content

print(point2_summary)

- **Time‑saved commutes & fewer interruptions:** Workers gain 30–60 min daily, boosting productivity (Gartner 2023: 77 % report higher output).  
- **Flexible schedules & deep‑work focus:** “No‑meeting” days and compressed weeks (e.g., Basecamp, Berlin analytics team) cut delivery times by ~12–18 % while improving work‑life balance.


In [ ]:
# Step 4 – Add a Call-to-Action

call_to_action_prompt = "Write a closing call-to-action encouraging companies to adopt remote work policies."

call_to_action = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": outline_prompt},
        {"role": "assistant", "content": outline},
        {"role": "user", "content": expand_prompt},
        {"role": "assistant", "content": point2_explaination},
        {"role": "user", "content": summary_prompt},
        {"role": "assistant", "content": point2_summary},
        {"role": "user", "content": call_to_action_prompt},
    ]
).choices[0].message.content

print(call_to_action)

**Ready to transform your workplace?**  
Adopting a remote‑work policy isn’t just a trend—it’s a strategic investment in talent, productivity, and sustainability. Start with a pilot: select a diverse team, set clear performance metrics, and provide the tools for seamless collaboration. Measure results over three months and let data guide your expansion. By embracing flexible work, you’ll attract top talent, lower overhead, and create a happier, more resilient workforce. **Take the first step today—design your remote‑work framework, launch the pilot, and watch your organization thrive.**


## **6) Using an LLM to provide Prompt**

Instead of framing the prompt ourself, we can ask an LLM to create a detailed prompt that can be used to get the desired response.

**Example Scenario:** Suppose we want to generate a professional email, but instead of manually writing the prompt, we first ask the LLM to create a better prompt for us.


In [ ]:
# Function to generate prompt

def generate_prompt(input):
    prompt = f"""For the given request create a detailed prompt to be fed to an LLM for response generation.
    Provide only the prompt and not the LLM response.
    Request: {input}"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content


In [ ]:
request = "write a professional email requesting a meeting with a client next week"
new_prompt = generate_prompt(request)

print(new_prompt)

**Prompt for the LLM**

You are an experienced business professional tasked with drafting a concise, courteous, and persuasive email to a client, requesting a meeting to be held sometime next week. Follow the guidelines below and include all required elements. Do **not** add any additional commentary beyond the email itself.

---

### Email Requirements
1. **Subject Line** – Clear and action‑oriented, indicating a meeting request.
2. **Greeting** – Use the client’s name (e.g., “Dear [Client Name],”).
3. **Opening Sentence** – Briefly reference a recent interaction or shared context (e.g., a previous call, project update, or mutual interest) to personalize the message.
4. **Purpose** – State that you would like to schedule a meeting to discuss **[specific topic or project]**.
5. **Proposed Timeframe** – Offer a range of days/times for the meeting next week (e.g., “Monday – Wednesday, between 10:00 AM and 3:00 PM”), and invite the client to suggest alternatives if none are convenient.
6.

In [ ]:
email_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": new_prompt}
    ]
)

print(email_response.choices[0].message.content)

Subject: Meeting Request – Discuss [Specific Topic or Project] Next Week  

Dear [Client Name],

I enjoyed our recent call on the project’s progress and appreciated your insights on the upcoming milestones.  

I would like to schedule a meeting next week to discuss **[Specific Topic or Project]** in more detail. Could we meet on **[Proposed Days]**, between **[Proposed Time Window]**? If those times are inconvenient, please let me know alternative slots that work for you.  

My preferred format is a **[Preferred Format]**, and I will send a calendar invitation once we confirm a time.  

During our conversation, we can review current results, explore new opportunities for optimization, and address any questions you may have, ensuring the project stays on track and delivers maximum value for your team.  

Thank you for considering this request. I look forward to connecting and advancing our collaboration.  

Best regards,  

[Your Name]  
[Your Title]  
[Your Company]  
Phone: [Your Phon

## **7) JSON Prompting**

**JSON Prompting** is a technique where you instruct an AI model to return its output in **JSON (JavaScript Object Notation)** format — a structured, machine-readable way of representing data.

Instead of producing free-flowing text, the model outputs information in a structured schema (with keys and values).

---

**Why JSON Prompting?**

- Ensures **consistent structure** for downstream systems.

- Makes it easy to **parse and integrate outputs** into software, dashboards, or APIs.

- Useful in **automation, reporting, and data pipelines**.

---

**Why it Works?**

The effectiveness of JSON Prompting it leverages how LLMs actually learn. During training, these models encounter millions of examples of structured data: API responses, configuration files, database schemas, and code documentation. JSON patterns are deeply embedded in their understanding.

JSON Prompting transforms AI from an unpredictable tool into a reliable system component. Teams can build automated workflows knowing that outputs will consistently match expected formats, reducing the need for complex error handling and manual intervention.

---

Consider the following example:

**Traditional Prompt:**
```
"Analyze this customer review and tell me about the sentiment"
```

**JSON Prompt:**
```
{
  "task": "sentiment_analysis",
  "input": "The product exceeded my expectations!",
  "output_format": {
    "sentiment": "positive|negative|neutral",
    "confidence": "0.0-1.0",
    "key_phrases": ["array", "of", "strings"],
    "summary": "brief explanation"
  }
}
```


In [ ]:
# Json Prompt example

json_prompt = """{
  "task": "sentiment_analysis",
  "input": "The product exceeded my expectations!",
  "output_format": {
    "sentiment": "positive|negative|neutral",
    "confidence": "0.0-1.0",
    "key_phrases": ["array", "of", "strings"],
    "summary": "brief explanation"
  }
}"""

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": json_prompt}
    ]
)

In [ ]:
output = response.choices[0].message.content
print(output)                       # <-- the output looks json but it is still a string

{
  "sentiment": "positive",
  "confidence": 0.99,
  "key_phrases": [
    "exceeded my expectations",
    "product"
  ],
  "summary": "The user expresses strong satisfaction, indicating the product performed better than anticipated."
}


In [ ]:
type(output)

str

In [ ]:
print('\n'.join(output.split('\n')[1:-1]))

  "sentiment": "positive",
  "confidence": 0.99,
  "key_phrases": [
    "exceeded my expectations",
    "product"
  ],
  "summary": "The user expresses strong satisfaction, indicating the product performed better than anticipated."


In [ ]:
# Convert the output string to Json (Python dictionary)

import json

try:
    parsed_output = json.loads(output)
except:
    parsed_output = json.loads('\n'.join(output.split('\n')[1:-1]))

type(parsed_output)

dict

In [ ]:
parsed_output

{'sentiment': 'positive',
 'confidence': 0.99,
 'key_phrases': ['exceeded my expectations', 'product'],
 'summary': 'The user expresses strong satisfaction, indicating the product performed better than anticipated.'}

In [ ]:
# Now you can access fields like a dictionary
print(parsed_output["sentiment"])
print(parsed_output["confidence"])
print(parsed_output["key_phrases"])
print(parsed_output["summary"])

positive
0.99
['exceeded my expectations', 'product']
The user expresses strong satisfaction, indicating the product performed better than anticipated.


## **8) Dynamic Prompt Generation Using Jinja Templates**

**Jinja** (commonly Jinja2) is a templating engine for Python that allows you to create dynamic text or documents by combining static text with variables and logic.

In simple terms, Jinja is a tool that lets you insert data and logic into text templates to generate final output dynamically.

**Basic Examples for Jinja Template**

**Example 1:** Variable substitution

In [ ]:
# Define a Jinja Template

from jinja2 import Template

template_str = """
Hello {{ name }}!

{{ message }}

Regards,
Support Team
"""

template = Template(template_str)

In [ ]:
# Render the Template
output = template.render(name="Aman", message="Thanks for visiting")
print(output)


Hello Aman!

Thanks for visiting

Regards,
Support Team


In [ ]:
# Render the Template for different inputs
output = template.render(name="Rashi", message="We appreciate your feedback")
print(output)


Hello Rashi!

We appreciate your feedback

Regards,
Support Team


**Example 2:** If-else condition

In [ ]:
# Define a Jinja Template

from jinja2 import Template

template_str = """
Hello {{ name }}!

{% if sentiment == "positive" %}
We're glad to hear that you had a great experience!

{% elif sentiment == "negative" %}
We're sorry your experience didn't go as expected. Thanks for bringing this to our attention. We will look into it right away.

{% else %}
We appreciate your feedback.

{% endif %}

Sincerely,
Support Team
"""

template = Template(template_str)

In [ ]:
# Render the Template
output = template.render(name="Joy", sentiment="positive")
print(output)


Hello Joy!


We're glad to hear that you had a great experience!



Sincerely,
Support Team


In [ ]:
# Render the Template for different inputs
output = template.render(name="Rashi", sentiment="negative")
print(output)


Hello Rashi!


We're sorry your experience didn't go as expected. Thanks for bringing this to our attention. We will look into it right away.



Sincerely,
Support Team


**Example 3: Jinja + LLM Call**

In [ ]:
# Define a Jinja Template

from jinja2 import Template

template_str = """
You are an expert {{ domain }} consultant.

Task:
{{ task }}

Context:
{{ context }}

Instructions:
{% if tone == "formal" %}
- Use professional language
{% else %}
- Use simple and friendly language
{% endif %}
- Keep the response under {{ max_words }} words
"""

In [ ]:
# Render the Template

template = Template(template_str)

prompt = template.render(
    domain="HR",
    task="Suggest employee engagement activities",
    context="Remote teams are feeling disconnected",
    tone="formal",
    max_words=100
)

print(prompt)


You are an expert HR consultant.

Task:
Suggest employee engagement activities

Context:
Remote teams are feeling disconnected

Instructions:

- Use professional language

- Keep the response under 100 words


In [ ]:
# Send Prompt to LLM

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

**Engagement activities for remote teams**

1. **Weekly virtual coffee chats** – informal, 15‑minute video calls to catch up.  
2. **Skill‑sharing webinars** – team members present on topics of interest.  
3. **Gamified wellness challenges** – step counts, hydration streaks with small rewards.  
4. **Structured 15‑minute check‑ins** – quick stand‑ups focused on wins, blockers, and support.  
5. **Rotating virtual happy hours** – host by a different team member each week.  
6. **Collaborative retrospectives** – review projects together, celebrating successes.  
7. **Online hackathons** – short, themed innovation sprints with prizes.  
8. **Peer‑recognition dashboards** – public kudos for accomplishments.  
9. **Virtual book clubs** – read and discuss relevant literature monthly.  
10. **Wellness webinars** – mindfulness, ergonomics, or mental‑health sessions.


# II. Practice Task

(Ungraded)

Create an **AI System** that can respond to customer and internal staff queries for a Flower Delivery Network.

<br>

<img src='https://drive.google.com/uc?id=1SFcrbz9nVcNHtMKUqQ18uBeKx1srOrY8' width=1000px>


**Intermediate steps:**

- **Create three different AI assistants**, each specializing in a different function of the flower delivery business:

    - **Order Support Assistant** – Handles customer issues like order status, delivery delays, bouquet customization, payment queries.

    - **Floral Expert Assistant** – Answers questions on flower types, care instructions, bouquet recommendations, seasonal availability.

    - **Logistics & Operations Assistant** – Handles driver routing, warehouse issues, delivery time estimates, packaging guidelines.

    (Hint: Use Role-based System prompts. Try including Jinja templates.)

- **Route incoming user queries** to the correct assistant based on the nature of the question –
Order Support / Floral Expert / Operations.

    (Hint: Use Few-shot examples + JSON classification prompting to determine the category.)

- **Test the system** using sample customer and staff queries, for example:

    - *Can I change the delivery time for my bouquet?*

    - *Which flowers last longest in hot weather?*

    - *A driver is stuck in traffic—what should be the updated delivery estimate?*


In [ ]:
import json

# Define the JSON schema for classification
classification_schema = """
{
  "query_category": "Order Support|Floral Expert|Logistics & Operations",
  "reasoning": "brief explanation of why this category was chosen"
}
"""

# Updated classification_prompt_template (only instructions and schema)
classification_prompt_template = f"""
Classify the following user query into one of these categories: 'Order Support', 'Floral Expert', or 'Logistics & Operations'.
Provide your response in JSON format as specified by the following schema:
{classification_schema}
"""

# Few-shot examples (as a list of dicts for clarity)
few_shot_examples = [
    {
        "user": "Can I change the delivery time for my bouquet?",
        "assistant": "{ \"query_category\": \"Order Support\", \"reasoning\": \"The query is about modifying an existing order's delivery details.\" }"
    },
    {
        "user": "Which flowers last longest in hot weather?",
        "assistant": "{ \"query_category\": \"Floral Expert\", \"reasoning\": \"The query is asking for advice on flower types and their suitability for specific conditions.\" }"
    },
    {
        "user": "A driver is stuck in traffic—what should be the updated delivery estimate?",
        "assistant": "{ \"query_category\": \"Logistics & Operations\", \"reasoning\": \"The query relates to real-time delivery issues and operational adjustments.\" }"
    },
    {
        "user": "My payment didn't go through, can you help?",
        "assistant": "{ \"query_category\": \"Order Support\", \"reasoning\": \"This query is about a payment issue related to an order.\" }"
    },
    {
        "user": "What are some good flower recommendations for a birthday?",
        "assistant": "{ \"query_category\": \"Floral Expert\", \"reasoning\": \"This query asks for bouquet recommendations.\" }"
    },
    {
        "user": "How do I handle a bulk order for a corporate event?",
        "assistant": "{ \"query_category\": \"Logistics & Operations\", \"reasoning\": \"The query is about managing a large order, which falls under operational planning.\" }"
    },
    {
        "user": "I want to cancel my order.",
        "assistant": "{ \"query_category\": \"Order Support\", \"reasoning\": \"The query is about cancelling an order.\" }"
    },
    {
        "user": "What flowers are in season in spring?",
        "assistant": "{ \"query_category\": \"Floral Expert\", \"reasoning\": \"The query asks about seasonal availability of flowers.\" }"
    },
    {
        "user": "What are the guidelines for packaging delicate flowers?",
        "assistant": "{ \"query_category\": \"Logistics & Operations\", \"reasoning\": \"The query is about packaging procedures.\" }"
    }
]

def classify_query(user_query):
    messages = [
        {"role": "system", "content": classification_prompt_template}
    ]

    for example in few_shot_examples:
        messages.append({"role": "user", "content": example["user"]})
        messages.append({"role": "assistant", "content": example["assistant"]})

    messages.append({"role": "user", "content": user_query}) # The actual query to classify

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages, # Pass the structured messages
        response_format={"type": "json_object"} # Requesting JSON output directly
    )

    # Attempt to parse the JSON response
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        print(f"Raw response content: {response.choices[0].message.content}")
        return {"query_category": "Unknown", "reasoning": "Failed to parse JSON response"}


In [ ]:
# Test the classification system with sample queries
sample_queries = [
    "Can I change the delivery time for my bouquet?",
    "Which flowers last longest in hot weather?",
    "A driver is stuck in traffic—what should be the updated delivery estimate?",
    "My credit card was charged twice for order #12345.",
    "What are the best flowers for a low-light apartment?",
    "How do we optimize delivery routes for efficiency?",
    "I need to know the status of my order from yesterday.",
    "Do you have any pet-safe flowers?",
    "What's the process for reporting a damaged shipment?"
]

# Map categories to their respective system prompts
assistant_prompts = {
    "Order Support": order_support_system_prompt,
    "Floral Expert": floral_expert_system_prompt,
    "Logistics & Operations": logistics_system_prompt
}

# Function to get assistant response
def get_assistant_response(category, user_query):
    system_prompt = assistant_prompts.get(category)
    if not system_prompt:
        return "No specific assistant found for this category."

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ]
    )
    return response.choices[0].message.content


print("--- Testing the AI System ---")
for query in sample_queries:
    print(f"\nUser Query: {query}")
    classification = classify_query(query)
    category = classification.get("query_category", "Unknown")
    reasoning = classification.get("reasoning", "N/A")

    print(f"Classified as: {category} (Reasoning: {reasoning})")
    assistant_response = get_assistant_response(category, query)
    print(f"Assistant's Response: {assistant_response}")


--- Testing the AI System ---

User Query: Can I change the delivery time for my bouquet?
Classified as: Order Support (Reasoning: The user is requesting to modify the delivery time of an existing order.)
Assistant's Response: Sure! Whether you can adjust the delivery time depends on where your order is in the process.

| Order stage | Can you change the time? | How to do it |
|-------------|------------------------|--------------|
| **Order placed, but not yet processed** | Yes | Log in to the app or website, open the order, tap “Change delivery time”, and pick a new slot. |
| **Order processed & scheduled but not shipped** | Usually yes | Use the same “Change delivery time” link, or call our 24 / 7 support line (1‑800‑FLOWERS) and we’ll reschedule for you. |
| **Order already shipped / on the way** | No | We can’t move the shipment, but if you need a different day, let us know and we’ll see if we can deliver the next available slot or offer a refund. |

If you’re unsure of the curren

In [ ]:
# YOUR CODE HERE...

In [ ]:
# Import Jinja2 Template for dynamic prompt generation
from jinja2 import Template

# Define a base Jinja template for AI assistants
assistant_template_str = """
You are an AI assistant specialized in the field of {{ domain }}.
Your role is to act as a {{ role }} for a flower delivery network.

Here are your specific instructions:
{{ instructions }}

Respond concisely and helpfully, always staying in character as a {{ role }}.
If you do not know the answer, politely state that you cannot fulfill the request.
"""

# Create the Jinja Template object
assistant_template = Template(assistant_template_str)


In [ ]:
# 1. Order Support Assistant System Prompt
order_support_instructions = """
- Handle customer issues related to order status, delivery delays, bouquet customization, and payment queries.
- Provide clear and empathetic responses.
- If an order detail needs to be changed, instruct the user on the steps to take or inform them if it's no longer possible.
"""

order_support_system_prompt = assistant_template.render(
    domain="customer service and order management",
    role="Order Support Assistant",
    instructions=order_support_instructions
)

print("--- Order Support Assistant System Prompt ---")
print(order_support_system_prompt)

# 2. Floral Expert Assistant System Prompt
floral_expert_instructions = """
- Answer questions on flower types, care instructions, bouquet recommendations, and seasonal availability.
- Offer creative and practical advice for flower care and selection.
- Provide details on specific flower characteristics and symbolism.
"""

floral_expert_system_prompt = assistant_template.render(
    domain="floriculture and floral design",
    role="Floral Expert Assistant",
    instructions=floral_expert_instructions
)

print("\n--- Floral Expert Assistant System Prompt ---")
print(floral_expert_system_prompt)

# 3. Logistics & Operations Assistant System Prompt
logistics_instructions = """
- Handle queries regarding driver routing, warehouse issues, delivery time estimates, and packaging guidelines.
- Provide accurate and timely information on operational processes.
- Advise on best practices for efficient delivery and handling of flowers.
"""

logistics_system_prompt = assistant_template.render(
    domain="logistics and supply chain management",
    role="Logistics & Operations Assistant",
    instructions=logistics_instructions
)

print("\n--- Logistics & Operations Assistant System Prompt ---")
print(logistics_system_prompt)


--- Order Support Assistant System Prompt ---

You are an AI assistant specialized in the field of customer service and order management.
Your role is to act as a Order Support Assistant for a flower delivery network.

Here are your specific instructions:

- Handle customer issues related to order status, delivery delays, bouquet customization, and payment queries.
- Provide clear and empathetic responses.
- If an order detail needs to be changed, instruct the user on the steps to take or inform them if it's no longer possible.


Respond concisely and helpfully, always staying in character as a Order Support Assistant.
If you do not know the answer, politely state that you cannot fulfill the request.

--- Floral Expert Assistant System Prompt ---

You are an AI assistant specialized in the field of floriculture and floral design.
Your role is to act as a Floral Expert Assistant for a flower delivery network.

Here are your specific instructions:

- Answer questions on flower types, car

In [ ]:
# YOUR CODE HERE...

In [ ]:
# YOUR CODE HERE...

In [ ]:
# YOUR CODE HERE...

**References:**

- [Prompt Engineering Guide](https://www.promptingguide.ai/)

- [Is JSON Prompting a Good Strategy?](https://blog.promptlayer.com/is-json-prompting-a-good-strategy/)

- [Collection of Work Prompts](https://www.aiforwork.co/)

---

<center>$END$</center>

---